# Event table and first momentum proxy

This notebook turns the FIFA timeline for Argentina vs Egypt into a tidy event table, assigns a transparent event-based threat score, and plots a first open momentum proxy with hydration breaks.

Important framing: this is **not** FIFA's official Match Momentum model. It is an open proxy built from public timeline events.

In [ ]:
import json
import math
import os
import re
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MPL_CACHE_DIR = PROJECT_ROOT / ".matplotlib-cache"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

RAW_DIR = PROJECT_ROOT / "data/raw"
PROCESSED_DIR = PROJECT_ROOT / "data/processed"
FIGURES_DIR = PROJECT_ROOT / "reports/figures"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

MATCH_ID = "400021528"
TIMELINE_PATH = RAW_DIR / "argentina_egypt_400021528_timeline.json"
LIVE_PATH = RAW_DIR / "argentina_egypt_400021528_live.json"

timeline = json.loads(TIMELINE_PATH.read_text())
live = json.loads(LIVE_PATH.read_text())

## 1. Helpers

In [ ]:
def localized_description(items, default=""):
    if not items:
        return default
    return items[0].get("Description", default)

def event_text(event):
    return " ".join(description.get("Description", "") for description in event.get("EventDescription", []))

def parse_match_minute(value):
    if value is None:
        return None
    text = str(value).replace("'", "")
    if "+" in text:
        base, added = text.split("+", 1)
        return float(base) + float(added)
    try:
        return float(text)
    except ValueError:
        return None

home_team = live["HomeTeam"]
away_team = live["AwayTeam"]
team_lookup = {
    home_team["IdTeam"]: {
        "side": "home",
        "name": home_team["ShortClubName"],
        "abbr": home_team["Abbreviation"],
        "sign": 1,
    },
    away_team["IdTeam"]: {
        "side": "away",
        "name": away_team["ShortClubName"],
        "abbr": away_team["Abbreviation"],
        "sign": -1,
    },
}

home_team["ShortClubName"], away_team["ShortClubName"], team_lookup

## 2. Flatten the FIFA timeline

One row per event. We keep original FIFA identifiers and add normalized fields used by the proxy.

In [ ]:
rows = []
for event in timeline.get("Event", []):
    team = team_lookup.get(event.get("IdTeam"), {})
    rows.append({
        "event_id": event.get("EventId"),
        "timestamp": event.get("Timestamp"),
        "match_minute_label": event.get("MatchMinute"),
        "match_minute": parse_match_minute(event.get("MatchMinute")),
        "period": event.get("Period"),
        "type_id": event.get("Type"),
        "event_type": localized_description(event.get("TypeLocalized"), "Unknown"),
        "description": event_text(event),
        "team_id": event.get("IdTeam"),
        "team": team.get("name"),
        "team_abbr": team.get("abbr"),
        "side": team.get("side"),
        "x": event.get("PositionX"),
        "y": event.get("PositionY"),
        "goal_x": event.get("GoalGatePositionX"),
        "goal_y": event.get("GoalGatePositionY"),
        "home_goals": event.get("HomeGoals"),
        "away_goals": event.get("AwayGoals"),
    })

events = pd.DataFrame(rows).sort_values(["match_minute", "timestamp", "event_id"], na_position="last")
events.head(10)

In [ ]:
events["event_type"].value_counts()

## 3. Hydration intervals

FIFA marks hydration breaks as `Delay` events followed by `Resume` events. These intervals are not inputs into the momentum score; they are annotations for analysis.

In [ ]:
hydration_intervals = []
open_break = None

for _, event in events.iterrows():
    text = f"{event['event_type']} {event['description']}".lower()
    if event["event_type"] == "Delay" and "hydration break" in text:
        open_break = event
    elif open_break is not None and event["event_type"] == "Resume":
        hydration_intervals.append({
            "start_minute": open_break["match_minute"],
            "start_label": open_break["match_minute_label"],
            "start_timestamp": open_break["timestamp"],
            "end_minute": event["match_minute"],
            "end_label": event["match_minute_label"],
            "end_timestamp": event["timestamp"],
            "period": open_break["period"],
        })
        open_break = None

hydration = pd.DataFrame(hydration_intervals)
hydration

## 4. Transparent event scoring

The official FIFA graphic is described as a possession-value model. Public FIFA timeline data does not include every pass/carry, so this proxy uses only visible events.

Scoring assumptions:

- Goals and penalties create large attacking threat.
- Shots are weighted by how close their recorded location is to either goal.
- Corners create moderate pressure.
- Fouls are credited to the opponent, because the event team is usually the team committing the foul.
- Goal-prevention events are credited to the opponent, because they usually indicate the defending goalkeeper/team had to prevent a chance.
- Cards, substitutions, delays, resumes, starts, and ends are annotations, not threat.

In [ ]:
def danger_from_location(x):
    if pd.isna(x):
        return 0.0
    # FIFA timeline coordinates are normalized 0-100. Without attacking direction
    # for every event, closeness to either goal is a useful first approximation.
    return max(0.0, min(1.0, 1.0 - min(float(x), 100.0 - float(x)) / 50.0))

def opponent_team_id(team_id):
    ids = list(team_lookup)
    if team_id == ids[0]:
        return ids[1]
    if team_id == ids[1]:
        return ids[0]
    return None

def score_event(row):
    event_type = row["event_type"]
    team_id = row["team_id"]
    x = row["x"]
    danger = danger_from_location(x)

    attacking_team_id = team_id
    score = 0.0
    reason = "annotation/no threat"

    if event_type == "Goal!":
        score = 5.0
        reason = "goal"
    elif event_type == "Penalty Awarded":
        score = 3.0
        reason = "penalty awarded"
    elif event_type == "Attempt at Goal":
        score = 1.2 + 1.4 * danger
        reason = "shot weighted by location"
    elif event_type == "Corner":
        score = 0.7
        reason = "corner pressure"
    elif event_type == "Goal Prevention":
        attacking_team_id = opponent_team_id(team_id)
        score = 0.9
        reason = "save/goal prevention credited to opponent attack"
    elif event_type == "Foul":
        attacking_team_id = opponent_team_id(team_id)
        score = 0.1 + 0.5 * danger
        reason = "foul credited to opponent, boosted near goal"
    elif event_type == "Offside":
        score = 0.2
        reason = "attacking offside"

    attacking_team = team_lookup.get(attacking_team_id, {})
    signed_score = score * attacking_team.get("sign", 0)

    return pd.Series({
        "attacking_team_id": attacking_team_id,
        "attacking_team": attacking_team.get("name"),
        "attacking_abbr": attacking_team.get("abbr"),
        "threat_score": score,
        "signed_threat": signed_score,
        "location_danger": danger,
        "score_reason": reason,
    })

events = pd.concat([events, events.apply(score_event, axis=1)], axis=1)
events.loc[events["threat_score"] > 0, ["match_minute_label", "event_type", "team_abbr", "attacking_abbr", "threat_score", "score_reason", "description"]].head(20)

## 5. Build the momentum curve

For each minute, compute the exponentially decayed recent threat difference:

`Argentina threat - Egypt threat`

A positive value favors Argentina. A negative value favors Egypt.

In [ ]:
MAX_MINUTE = math.ceil(events["match_minute"].dropna().max())
GRID_STEP = 0.25
HALF_LIFE_MINUTES = 4.0

grid = pd.DataFrame({"minute": [round(i * GRID_STEP, 2) for i in range(int(MAX_MINUTE / GRID_STEP) + 1)]})
scored_events = events[(events["threat_score"] > 0) & events["match_minute"].notna()].copy()

def decayed_momentum_at(minute):
    prior = scored_events[scored_events["match_minute"] <= minute]
    if prior.empty:
        return 0.0
    age = minute - prior["match_minute"]
    weights = 0.5 ** (age / HALF_LIFE_MINUTES)
    return float((prior["signed_threat"] * weights).sum())

grid["raw_momentum"] = grid["minute"].apply(decayed_momentum_at)
max_abs = grid["raw_momentum"].abs().max()
grid["momentum"] = 100 * grid["raw_momentum"] / max_abs if max_abs else 0.0
grid.head(), grid.tail()

## 6. Save event and momentum tables

In [ ]:
events_path = PROCESSED_DIR / "argentina_egypt_400021528_events.csv"
momentum_path = PROCESSED_DIR / "argentina_egypt_400021528_momentum.csv"
hydration_path = PROCESSED_DIR / "argentina_egypt_400021528_hydration_breaks.csv"

events.to_csv(events_path, index=False)
grid.to_csv(momentum_path, index=False)
hydration.to_csv(hydration_path, index=False)

events_path, momentum_path, hydration_path

## 7. Plot first momentum proxy

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(grid["minute"], grid["momentum"], color="#1f77b4", linewidth=2.5, label="Open momentum proxy")
ax.axhline(0, color="#1f2937", linewidth=1)
ax.fill_between(grid["minute"], 0, grid["momentum"], where=grid["momentum"] >= 0, color="#75aadb", alpha=0.25)
ax.fill_between(grid["minute"], 0, grid["momentum"], where=grid["momentum"] < 0, color="#d4af37", alpha=0.25)

for _, interval in hydration.iterrows():
    ax.axvspan(interval["start_minute"], interval["end_minute"], color="#8b5cf6", alpha=0.18)
    ax.text(
        (interval["start_minute"] + interval["end_minute"]) / 2,
        95,
        "Hydration",
        ha="center",
        va="top",
        fontsize=9,
        color="#4c1d95",
    )

goals = events[events["event_type"] == "Goal!"].copy()
for _, goal in goals.iterrows():
    color = "#1f77b4" if goal["attacking_abbr"] == home_team["Abbreviation"] else "#d4af37"
    ax.axvline(goal["match_minute"], color=color, linestyle="--", linewidth=1.2, alpha=0.75)
    ax.text(goal["match_minute"], -95, f"Goal {goal['attacking_abbr']} {goal['match_minute_label']}", rotation=90, va="bottom", ha="right", fontsize=8, color=color)

ax.set_title("Argentina vs Egypt, WC26 Round of 16\nOpen Event-Based Momentum Proxy with Hydration Breaks", fontsize=15, weight="bold")
ax.set_xlabel("Match minute")
ax.set_ylabel("Momentum proxy (-100 Egypt, +100 Argentina)")
ax.set_xlim(0, MAX_MINUTE)
ax.set_ylim(-105, 105)
ax.text(0.01, 0.97, "Argentina pressure", transform=ax.transAxes, color="#1f77b4", va="top", fontsize=10)
ax.text(0.01, 0.03, "Egypt pressure", transform=ax.transAxes, color="#9a6b00", va="bottom", fontsize=10)
ax.legend(loc="upper right")
fig.tight_layout()

figure_path = FIGURES_DIR / "argentina_egypt_400021528_momentum_proxy.png"
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
figure_path

## 8. First before/after break summary

This is a first descriptive summary. We compare the average momentum in the five minutes before each hydration break with the five minutes after play resumes.

In [ ]:
summary_rows = []
WINDOW = 5.0

for index, interval in hydration.iterrows():
    before = grid[(grid["minute"] >= interval["start_minute"] - WINDOW) & (grid["minute"] < interval["start_minute"])]
    after = grid[(grid["minute"] > interval["end_minute"]) & (grid["minute"] <= interval["end_minute"] + WINDOW)]
    before_avg = before["momentum"].mean()
    after_avg = after["momentum"].mean()
    summary_rows.append({
        "break_number": index + 1,
        "interval": f"{interval['start_label']} to {interval['end_label']}",
        "before_avg_momentum": before_avg,
        "after_avg_momentum": after_avg,
        "delta_after_minus_before": after_avg - before_avg,
        "before_leader": "Argentina" if before_avg > 0 else "Egypt",
        "after_leader": "Argentina" if after_avg > 0 else "Egypt",
    })

break_summary = pd.DataFrame(summary_rows)
break_summary_path = PROCESSED_DIR / "argentina_egypt_400021528_break_summary.csv"
break_summary.to_csv(break_summary_path, index=False)
break_summary